# Chronos-2 LoRA fine-tuning on Dataset E

Ordinary LoRA fine-tuning of `amazon/chronos-2` on the project's long financial
panel, so the adapted model can later be run on the 12 official M6 origins and
compared with the existing zero-shot Chronos-2 result.

| | |
|---|---|
| Base model | `amazon/chronos-2`, the same checkpoint as the zero-shot run |
| Training data | Dataset E, extended EODHD adjusted closes, 100 M6 assets, 2000 to 2023 |
| Representation | one multivariate task, `(1, 100, time)`, assets modelled jointly |
| Context / horizon | 512 past weekdays, 20 forecast weekdays |
| Adapter | LoRA rank 8, alpha 16, official default target modules |
| Loss | the official Chronos-2 quantile loss, unchanged |

The split is chronological. Training targets never run past 2022-02-04. One
internal validation window has its context ending 2022-02-04 and its target
covering the next 20 project weekdays, 2022-02-07 to 2022-03-04. The first M6
target is 2022-03-07, so the 12 scored rounds stay untouched.

This notebook trains and saves a checkpoint. It does not run M6 inference: that
is `chronos2_lora_m6.ipynb`, run afterwards.

Place Dataset E on Drive at the path shown in section 4 before running.

## 1. Install Chronos-2 and PEFT

In [ ]:
%pip install -q "chronos-forecasting==2.3.1" "transformers>=4.41,<5" "peft>=0.11"

In [ ]:
%pip uninstall -y torchao

## 2. Imports and configuration

The model, context length, horizon, asset order and missing-value convention are
carried over unchanged from the zero-shot Chronos-2 run, so the only difference
between that result and this one is the LoRA adapter.

In [ ]:
import inspect
import json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import Chronos2Pipeline

# --- Base model, unchanged from the zero-shot run ---------------------------
MODEL_ID = "amazon/chronos-2"

# --- Fixed experiment settings ----------------------------------------------
CONTEXT_LENGTH = 512          # also the minimum past length
MIN_PAST = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42

# --- LoRA configuration ------------------------------------------------------
LORA_RANK = 8
LORA_ALPHA = 16
LORA_TARGET_MODULES = None    # None = the official Chronos-2 defaults
LEARNING_RATE = 1e-5
MAX_STEPS = 1000
EVAL_EVERY = 100              # validate roughly every 100 steps where supported

# --- Chronological split, verified against Dataset E in section 7 ------------
TRAIN_TARGET_CEILING = "2022-02-04"   # no training target may fall after this
VALIDATION_ORIGIN = "2022-02-04"      # validation context ends here
VALIDATION_TARGET_START = "2022-02-07"
VALIDATION_TARGET_END = "2022-03-04"
FIRST_M6_TARGET = "2022-03-07"        # the holdout starts here and is untouched

# Official M6 asset order - used to verify the panel, not to reorder it.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Versions")
print(f"  chronos-forecasting : {version('chronos-forecasting')}")
print(f"  transformers        : {version('transformers')}")
print(f"  torch               : {torch.__version__}")
print(f"\nBase model : {MODEL_ID}")
print(f"LoRA       : rank {LORA_RANK}, alpha {LORA_ALPHA}, lr {LEARNING_RATE}, "
      f"max_steps {MAX_STEPS}, eval every {EVAL_EVERY}")
print(f"Shapes     : context {CONTEXT_LENGTH}, horizon {PREDICTION_LENGTH}, "
      f"{N_ASSETS} assets as one multivariate task")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Paths

Three separate Drive locations: the Dataset E input you place manually, the
saved LoRA checkpoint, and (used by the inference notebook, not here) the
fine-tuned M6 outputs. The shared M6 context folder is untouched by this
notebook.

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/HonoursResearch")

# 1. Dataset E input - PLACE THIS FILE HERE BEFORE RUNNING
DATASET_E_PATH = Path("/content/drive/MyDrive/HonoursResearch/datasets/extended_eodhd_ohlcv_2000_2023.csv")

# 2. Saved LoRA checkpoint, written by this notebook
LORA_CHECKPOINT_DIR = Path("/content/drive/MyDrive/HonoursResearch/checkpoints/chronos2_lora")

# 3. Fine-tuned M6 outputs, written later by chronos2_lora_m6.ipynb
M6_OUTPUT_DIR = Path("/content/drive/MyDrive/HonoursResearch/outputs/chronos2_lora/m6")

print("Dataset E (input, place manually):")
print(f"   {DATASET_E_PATH}")
print("LoRA checkpoint (written here):")
print(f"   {LORA_CHECKPOINT_DIR}")
print("Fine-tuned M6 outputs (written by the inference notebook):")
print(f"   {M6_OUTPUT_DIR}")

if not DATASET_E_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset E not found at:\n  {DATASET_E_PATH}\n\n"
        "Copy the extended EODHD adjusted-close panel there. In the repository "
        "it is:\n  Quasi-Average Reimplementation DRE-as-Stock/data/"
        "adjusted_close_panel.csv\n"
        "It must be the wide weekday panel: a 'date' column plus the 100 "
        "official M6 asset columns, covering roughly 2000-01-03 to 2023-02-03."
    )

LORA_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nDataset E found: {DATASET_E_PATH.stat().st_size / 1e6:.1f} MB")

## 5. GPU check

A GPU is required for fine-tuning. This stops rather than silently using CPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU (an L4 is recommended), then re-run from section 3."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU    : {GPU_NAME}")
print(f"memory : {GPU_MEMORY_GB:.1f} GB")

## 6. Require PEFT and LoRA support

Fine-tuning must adapt a small set of LoRA parameters, never the whole model. If
PEFT is missing, or the installed Chronos-2 build exposes no LoRA option, this
cell stops here rather than letting training fall back to full fine-tuning.

In [ ]:
try:
    import peft
    from peft import LoraConfig
except ImportError as exc:
    raise RuntimeError(
        "PEFT is not available, so LoRA fine-tuning cannot run. Install it with "
        "`%pip install peft>=0.11` and re-run. Training must NOT proceed without "
        "it: the run would silently become full-model fine-tuning."
    ) from exc

print(f"peft {peft.__version__}")

# The installed fit() must expose a LoRA switch. Inspect, never assume.
FIT_SIGNATURE = inspect.signature(Chronos2Pipeline.fit)
FIT_PARAMS = list(FIT_SIGNATURE.parameters)
print(f"\nChronos2Pipeline.fit parameters:\n  {FIT_PARAMS}")

LORA_HINTS = [p for p in FIT_PARAMS
              if any(k in p.lower() for k in ("lora", "peft", "adapter"))]
if not LORA_HINTS:
    raise RuntimeError(
        "Chronos2Pipeline.fit exposes no LoRA/PEFT parameter in this build:\n"
        f"  {FIT_PARAMS}\n"
        "Calling fit() now would fine-tune the FULL model, which is not the "
        "intended experiment. Stop and confirm the correct API before training."
    )
print(f"LoRA-related fit parameters found: {LORA_HINTS}")

## 7. Load Dataset E and build daily log returns

Dataset E ships adjusted closes, so returns are derived here using the project's
established convention: forward-fill only after each asset's genuine inception,
then `log(P[t] / P[t-1])`. Nothing is backfilled, interpolated, clipped,
winsorised or rescaled, and the genuine leading NaNs before an asset existed are
preserved for Chronos-2's own missing-value handling.

In [ ]:
panel = pd.read_csv(DATASET_E_PATH, parse_dates=["date"]).set_index("date")

assert list(panel.columns) == OFFICIAL_ASSET_ORDER, (
    "Dataset E columns are not the 100 official M6 assets in official order"
)
assert panel.index.is_monotonic_increasing, "Dataset E dates are not ascending"
assert not panel.index.duplicated().any(), "Dataset E has duplicate dates"
assert set(panel.index.dayofweek) <= {0, 1, 2, 3, 4}, "Dataset E is not a weekday panel"

# Forward-fill only after inception. pandas ffill never fills before the first
# valid value, so pre-inception history stays missing, exactly as intended.
filled = panel.ffill()
returns = np.log(filled / filled.shift(1))
returns = returns.where(filled.notna() & filled.shift(1).notna())

# Every remaining gap must be a genuine leading gap, never an interior one.
leading_total, interior_total = 0, 0
for symbol in returns.columns:
    values = returns[symbol].to_numpy()
    n_lead = int(np.argmax(~np.isnan(values))) if np.isnan(values[0]) else 0
    leading_total += n_lead
    interior_total += int(np.isnan(values[n_lead:]).sum())
assert interior_total == 0, (
    f"Dataset E produced {interior_total} interior NaNs; the panel is not the "
    "expected forward-filled weekday panel"
)

DATE_INDEX = returns.index
print(f"Dataset E: {panel.shape[0]} weekday rows x {panel.shape[1]} assets")
print(f"  {DATE_INDEX.min().date()} .. {DATE_INDEX.max().date()}")
print(f"  leading NaN (genuine pre-inception) : {leading_total:,}")
print(f"  interior NaN                        : {interior_total}")
print(f"  assets live on the last row         : {int(returns.iloc[-1].notna().sum())}/100")

## 8. Chronological split

The positions are derived from Dataset E's own date index rather than assumed,
and the boundaries are then checked explicitly.

The training array ends on the training ceiling, so no cutoff sampled inside it
can produce a target beyond that date. The validation array extends 20 weekdays
further, so its final window is exactly the validation target.

In [ ]:
def position_of(date_string: str) -> int:
    stamp = pd.Timestamp(date_string)
    if stamp not in DATE_INDEX:
        raise ValueError(f"{date_string} is not a row in Dataset E's calendar")
    return int(DATE_INDEX.get_loc(stamp))


train_end_pos = position_of(TRAIN_TARGET_CEILING)
validation_origin_pos = position_of(VALIDATION_ORIGIN)

# The 20 weekdays after the validation origin, taken from the actual index.
validation_target = DATE_INDEX[validation_origin_pos + 1:
                               validation_origin_pos + 1 + PREDICTION_LENGTH]
assert len(validation_target) == PREDICTION_LENGTH, (
    "Dataset E does not contain 20 weekdays after the validation origin"
)
assert validation_target[0] == pd.Timestamp(VALIDATION_TARGET_START), (
    f"validation target starts {validation_target[0].date()}, "
    f"expected {VALIDATION_TARGET_START}"
)
assert validation_target[-1] == pd.Timestamp(VALIDATION_TARGET_END), (
    f"validation target ends {validation_target[-1].date()}, "
    f"expected {VALIDATION_TARGET_END}"
)
# The scored M6 holdout must begin strictly after everything used here.
assert validation_target[-1] < pd.Timestamp(FIRST_M6_TARGET), (
    "the validation target overlaps the first M6 target"
)

validation_end_pos = validation_origin_pos + PREDICTION_LENGTH

# One multivariate task each: (1, 100, time), assets on the variate axis.
train_array = returns.to_numpy(dtype=np.float32).T[np.newaxis, :, :train_end_pos + 1]
validation_array = returns.to_numpy(dtype=np.float32).T[np.newaxis, :, :validation_end_pos + 1]

assert train_array.shape == (1, N_ASSETS, train_end_pos + 1)
assert validation_array.shape == (1, N_ASSETS, validation_end_pos + 1)
assert train_array.shape[2] >= MIN_PAST + PREDICTION_LENGTH, (
    "the training array is too short for even one context plus target"
)

print(f"training array   : {train_array.shape}  "
      f"{DATE_INDEX[0].date()} .. {DATE_INDEX[train_end_pos].date()}")
print(f"validation array : {validation_array.shape}  "
      f"{DATE_INDEX[0].date()} .. {DATE_INDEX[validation_end_pos].date()}")
print()
print(f"  training targets never pass : {TRAIN_TARGET_CEILING}")
print(f"  validation context ends     : {DATE_INDEX[validation_origin_pos].date()}")
print(f"  validation target           : {validation_target[0].date()} .. "
      f"{validation_target[-1].date()}  ({len(validation_target)} weekdays)")
print(f"  first M6 target (untouched) : {FIRST_M6_TARGET}")
print("\nBoundary checks passed.")

## 9. Load the base Chronos-2 model

The same checkpoint as the zero-shot run, loaded once.

In [ ]:
pipeline = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map="cuda")

QUANTILE_LEVELS = np.asarray(pipeline.quantiles, dtype=np.float64)
BASE_TOTAL_PARAMS = sum(p.numel() for p in pipeline.model.parameters())
BASE_TRAINABLE = sum(p.numel() for p in pipeline.model.parameters() if p.requires_grad)

print(f"Loaded: {MODEL_ID}")
print(f"  parameters      : {BASE_TOTAL_PARAMS:,} ({BASE_TOTAL_PARAMS / 1e6:.1f}M)")
print(f"  quantile levels : {len(QUANTILE_LEVELS)}")
print(f"  trainable before fine-tuning : {BASE_TRAINABLE:,} "
      f"({BASE_TRAINABLE / BASE_TOTAL_PARAMS:.1%})")

## 10. Map the configuration onto the installed `fit()`

Parameter names vary between Chronos-2 builds, so rather than assuming them this
cell matches each setting against the installed signature and reports exactly
what will be passed. Anything essential that cannot be matched stops the run,
with the real signature printed so the call can be corrected.

Everything not listed here is left at the library's own default, including the
quantile loss, which is not modified.

In [ ]:
# Each entry: intent -> (candidate parameter names, value, required?)
INTENT = [
    ("LoRA rank",          ["lora_rank", "lora_r", "r"],                      LORA_RANK,          True),
    ("LoRA alpha",         ["lora_alpha", "alpha"],                           LORA_ALPHA,         True),
    ("LoRA target modules",["lora_target_modules", "target_modules"],         LORA_TARGET_MODULES, False),
    ("learning rate",      ["learning_rate", "lr"],                           LEARNING_RATE,      True),
    ("max steps",          ["max_steps", "num_steps", "steps", "max_train_steps"], MAX_STEPS,    True),
    ("eval interval",      ["eval_every", "eval_steps", "evaluation_steps",
                            "validation_interval", "val_check_interval"],      EVAL_EVERY,        False),
    ("seed",               ["seed", "random_seed"],                           RANDOM_SEED,        False),
    ("context length",     ["context_length"],                                CONTEXT_LENGTH,     False),
    ("min past",           ["min_past"],                                      MIN_PAST,           False),
    ("prediction length",  ["prediction_length"],                             PREDICTION_LENGTH,  False),
    ("output dir",         ["output_dir", "save_dir", "checkpoint_dir"],       str(LORA_CHECKPOINT_DIR), False),
]

fit_kwargs, resolved, unmatched, missing_required = {}, [], [], []
for label, candidates, value, required in INTENT:
    name = next((c for c in candidates if c in FIT_PARAMS), None)
    if name is None:
        (missing_required if required else unmatched).append((label, candidates))
        continue
    if value is None:          # keep the library default
        resolved.append((label, name, "library default"))
        continue
    fit_kwargs[name] = value
    resolved.append((label, name, value))

# Enable LoRA explicitly if the build uses a boolean switch for it.
for flag in ("use_lora", "lora", "enable_lora", "use_peft"):
    if flag in FIT_PARAMS:
        fit_kwargs[flag] = True
        resolved.append(("enable LoRA", flag, True))
        break

print("Resolved fit() arguments:")
for label, name, value in resolved:
    print(f"  {label:<20} -> {name:<22} = {value}")
if unmatched:
    print("\nNot available in this build, library defaults will apply:")
    for label, candidates in unmatched:
        print(f"  {label:<20} (tried {candidates})")

if missing_required:
    raise RuntimeError(
        "These required settings have no matching parameter in the installed "
        "fit():\n"
        + "\n".join(f"  {label}: tried {c}" for label, c in missing_required)
        + f"\n\nActual signature:\n  {FIT_SIGNATURE}\n"
        "Correct the candidate names above before training."
    )

# The data arguments: first positional parameter carries the training data.
DATA_PARAM = next(p for p in FIT_PARAMS if p not in ("self", "cls"))

# Internal validation is REQUIRED. The official Chronos-2 fit() takes
# validation_inputs and uses the validation loss for best-model selection, so a
# run without it would silently keep the final checkpoint instead of the best.
VALIDATION_CANDIDATES = ["validation_inputs", "validation_data", "val_inputs",
                         "eval_inputs", "validation_set"]
VALIDATION_PARAM = next((c for c in VALIDATION_CANDIDATES if c in FIT_PARAMS), None)

print(f"\ntraining data parameter   : {DATA_PARAM}")
print(f"validation data parameter : {VALIDATION_PARAM or 'NONE FOUND'}")

if VALIDATION_PARAM is None:
    raise RuntimeError(
        "No validation-inputs parameter exists in the installed fit():\n"
        f"  tried {VALIDATION_CANDIDATES}\n"
        f"  actual signature: {FIT_SIGNATURE}\n\n"
        "Internal validation is required for this experiment: it is what drives "
        "best-model selection. Training must not proceed without it, because the "
        "run would keep the final checkpoint rather than the best one. Correct "
        "the candidate names above before training."
    )

## 11. Fine-tune

Runs the official `fit()` path with the resolved arguments. Immediately
afterwards the trainable parameters are counted: a LoRA run must leave the great
majority of the model frozen, so an unexpectedly large trainable share means the
run was not LoRA and the result must not be used.

In [ ]:
training_started = datetime.now(timezone.utc)

call_kwargs = dict(fit_kwargs)
# Required: resolved and checked in section 10.
call_kwargs[VALIDATION_PARAM] = validation_array

print(f"Calling fit() with {len(call_kwargs)} explicit arguments "
      f"(everything else at library defaults)\n")

fit_result = pipeline.fit(train_array, **call_kwargs)

# fit() may adapt in place or return a new pipeline; accept either.
tuned_pipeline = fit_result if isinstance(fit_result, Chronos2Pipeline) else pipeline

elapsed_minutes = (datetime.now(timezone.utc) - training_started).total_seconds() / 60
print(f"\nTraining finished in {elapsed_minutes:.1f} min")

# --- LoRA safety check -------------------------------------------------------
total = sum(p.numel() for p in tuned_pipeline.model.parameters())
trainable_params = [(n, p.numel()) for n, p in tuned_pipeline.model.named_parameters()
                    if p.requires_grad]
trainable = sum(n for _, n in trainable_params)
share = trainable / total if total else 0.0

print(f"\nparameters total     : {total:,}")
print(f"parameters trainable : {trainable:,} ({share:.2%})")

adapter_like = [n for n, _ in trainable_params
                if any(k in n.lower() for k in ("lora", "adapter"))]
non_adapter = [n for n, _ in trainable_params
               if not any(k in n.lower() for k in ("lora", "adapter"))]
print(f"adapter parameters   : {len(adapter_like)}")
print(f"other trainable      : {len(non_adapter)}")
if non_adapter:
    print(f"  first few: {non_adapter[:8]}")

assert trainable > 0, "No trainable parameters: fit() did not adapt anything."
assert share < 0.25, (
    f"{share:.1%} of parameters are trainable. A rank-{LORA_RANK} LoRA adapter "
    "should be a small fraction of the model, so this looks like full "
    "fine-tuning. Do not use this checkpoint."
)
assert adapter_like, (
    "No trainable parameter name contains 'lora' or 'adapter'. LoRA does not "
    "appear to have been applied. Do not use this checkpoint."
)
print("\nLoRA check passed: only a small adapter set is trainable.")

## 12. Save the fine-tuned model

One checkpoint at a stable path, overwritten on a rerun rather than accumulating
copies. Run metadata is saved beside it so the inference notebook and the later
analysis can confirm exactly what produced the forecasts.

In [ ]:
saver = next((getattr(tuned_pipeline, m) for m in
              ("save_pretrained", "save_adapter", "save")
              if hasattr(tuned_pipeline, m)), None)
if saver is None:
    raise RuntimeError(
        "The tuned pipeline exposes no save_pretrained/save_adapter/save method. "
        f"Available: {[m for m in dir(tuned_pipeline) if 'save' in m.lower()]}"
    )

saver(str(LORA_CHECKPOINT_DIR))

TRAINING_METADATA = {
    "base_model_id": MODEL_ID,
    "adapter": "lora",
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_target_modules": LORA_TARGET_MODULES or "library default",
    "learning_rate": LEARNING_RATE,
    "max_steps": MAX_STEPS,
    "eval_every": EVAL_EVERY,
    "random_seed": RANDOM_SEED,
    "context_length": CONTEXT_LENGTH,
    "min_past": MIN_PAST,
    "prediction_length": PREDICTION_LENGTH,
    "training_dataset": "Dataset E, extended EODHD adjusted closes, 100 M6 assets",
    "training_rows": int(train_array.shape[2]),
    "training_range": f"{DATE_INDEX[0].date()} .. {DATE_INDEX[train_end_pos].date()}",
    "train_target_ceiling": TRAIN_TARGET_CEILING,
    "validation_origin": VALIDATION_ORIGIN,
    "validation_target": f"{VALIDATION_TARGET_START} .. {VALIDATION_TARGET_END}",
    "internal_validation_parameter": VALIDATION_PARAM,
    "resolved_fit_arguments": {k: str(v) for k, v in call_kwargs.items()
                               if k != VALIDATION_PARAM},
    "trainable_parameters": int(trainable),
    "total_parameters": int(total),
    "trainable_share": round(share, 6),
    "training_minutes": round(elapsed_minutes, 1),
    "gpu": GPU_NAME,
    "chronos_forecasting": version("chronos-forecasting"),
    "peft": peft.__version__,
    "saved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
}
metadata_path = LORA_CHECKPOINT_DIR / "training_metadata.json"
metadata_path.write_text(json.dumps(TRAINING_METADATA, indent=2), encoding="utf-8")

saved = sorted(p.name for p in LORA_CHECKPOINT_DIR.iterdir())
print(f"Saved to {LORA_CHECKPOINT_DIR}")
for name in saved:
    print(f"  {name}")

## 13. Reload check

Confirms the saved checkpoint loads cleanly, so the inference notebook will not
fail on a broken save. No M6 inference happens here.

In [ ]:
del tuned_pipeline, fit_result
torch.cuda.empty_cache()

loader = getattr(Chronos2Pipeline, "from_pretrained")
try:
    reloaded = loader(str(LORA_CHECKPOINT_DIR), device_map="cuda")
except Exception:
    # Some builds save only the adapter, which loads onto the base model.
    print("Direct load failed; retrying as base model plus adapter.")
    reloaded = loader(MODEL_ID, device_map="cuda")
    attach = next((getattr(reloaded, m) for m in
                   ("load_adapter", "load_lora", "load_pretrained_adapter")
                   if hasattr(reloaded, m)), None)
    if attach is None:
        raise RuntimeError(
            "The checkpoint did not load directly and the pipeline exposes no "
            "load_adapter method. Record the exact save/load API before running "
            "the inference notebook."
        )
    attach(str(LORA_CHECKPOINT_DIR))

reloaded_quantiles = np.asarray(reloaded.quantiles, dtype=np.float64)
assert np.allclose(reloaded_quantiles, QUANTILE_LEVELS), (
    "the reloaded model reports different quantile levels"
)
print(f"Reloaded from {LORA_CHECKPOINT_DIR}")
print(f"  quantile levels : {len(reloaded_quantiles)} (unchanged)")
print(f"  parameters      : {sum(p.numel() for p in reloaded.model.parameters()):,}")
print("\nCheckpoint is loadable. Run chronos2_lora_m6.ipynb next.")

## 14. Summary

In [ ]:
print("Chronos-2 LoRA fine-tuning")
print(f"  base model   : {MODEL_ID}")
print(f"  adapter      : LoRA rank {LORA_RANK}, alpha {LORA_ALPHA}, "
      f"lr {LEARNING_RATE}, max_steps {MAX_STEPS}")
print(f"  trainable    : {trainable:,}/{total:,} ({share:.2%})")
print(f"  training data: Dataset E, {train_array.shape[2]} weekday rows, "
      f"{DATE_INDEX[0].date()} .. {DATE_INDEX[train_end_pos].date()}")
print(f"  validation   : context ends {VALIDATION_ORIGIN}, target "
      f"{VALIDATION_TARGET_START} .. {VALIDATION_TARGET_END}")
print(f"  runtime      : {elapsed_minutes:.1f} min on {GPU_NAME}")
print(f"  checkpoint   : {LORA_CHECKPOINT_DIR}")
print()
print(f"  Internal validation ran through the official fit() path via "
      f"'{VALIDATION_PARAM}', which drives best-model selection,")
print("  so the saved checkpoint is the best one by validation loss rather")
print("  than simply the last step.")
print()
print("  Next: chronos2_lora_m6.ipynb, which reads this checkpoint and runs the")
print("  12 official M6 origins. No scoring happens in either notebook.")